In [1]:
import numpy as np
from glob import glob
import os
import skimage as sk
from skan import draw
from skan.csr import skeleton_to_csgraph
from skan import Skeleton, summarize
import napari
import matplotlib.pyplot as plt
import pandas as pd

## Create Skeletons!

In [ ]:
cropped_imgs = sorted(glob("cropped_imgs/*.tif"))
cropped_imgs = list(map(sk.io.imread,cropped_imgs))

In [ ]:
equalized_hist = [sk.exposure.equalize_adapthist(img) for img in cropped_imgs]

In [ ]:
#create binary images using Yen's method
threshold_yen =[img > sk.filters.threshold_yen(img) for img in equalized_hist]

In [ ]:
#remove objects smaller than 30 pixels from binary images
filtered_objects = [sk.morphology.remove_small_objects(img,min_size=30) for img in threshold_yen]

In [ ]:
#create labeled images from filtered binary images
labeled_objects = [sk.morphology.label(img) for img in filtered_objects]

In [ ]:
#skeletonize labeled images
skeletons = [sk.morphology.skeletonize(img) for img in labeled_objects]

In [ ]:
#save skeleton images
files = sorted(glob("cropped_imgs/*.tif"))
path = "skeletons"
for i in range(len(skeletons)):
    name = os.path.basename(files[i])
    sk.io.imsave(os.path.join(path,name[:-4]+'_skeleton.tif'),skeletons[i])


In [ ]:
#save overlay images of skeletons on equalized hist images
save_path = 'skeletons'
for i in range(len(skeletons)):
    name = os.path.basename(files[i])
    fig,ax=plt.subplots()
    draw.overlay_skeleton_2d(equalized_hist[i],skeletons[i],image_cmap='Greys_r',dilate=1,axes=ax)
    fig.set_tight_layout(tight=True)
    plt.show(block=True)
    fig.savefig(os.path.join(save_path,name[:-4]+'_skeleton_overlay.png'),dpi=300)


### Analyze Skeletons of mito networks

In [ ]:
spacing_um = 0.07
branch_data = [summarize(Skeleton(skeleton,spacing=spacing_um),separator='_') for skeleton in skeletons]

In [ ]:
#assess dataframe
branch_data[9].tail()

In [ ]:
#quick visual of all the branch distances organized by branch type
branch_data.hist(column='branch_distance', by='branch_type', bins=100)

In [ ]:
#Quick visual of how the skeletons are labeled by branch type
draw.overlay_euclidean_skeleton_2d(equalized_hist[0], branch_data[0],
                                   skeleton_color_source='branch_type')

In [ ]:
files = sorted(glob("cropped_imgs/*.tif"))
names = [os.path.basename(file) for file in files]

In [ ]:
#add filename column to each dataframe and merge all dataframes into one
for name,df in zip(names,branch_data):
    df['filename'] = name

merged_df = pd.concat(branch_data)

In [ ]:
merged_df.to_csv('Output/skeleton_df.csv')

In [ ]:
import seaborn as sns

In [ ]:
#plot branch distances for junction to junction branches (branch_type 1) per condition; code from SKAN docs
#added a 'Condition' column based on filename
j2j=(merged_df[merged_df['branch_type']== 1].
     rename(columns={'branch_distance':
                     'branch distance (um)'}))
per_image = j2j.groupby('filename').median()
per_image['Condition'] = ['500 mM' if '500mm' in fn else '50 mM' for fn in per_image.index]

sns.stripplot(data=per_image,
              x='Condition',y='branch distance (um)',
              order=['50 mM','500 mM'],
              jitter=True)

In [ ]:
per_image.to_csv("Output/skeleton_df.csv")